# FRESEAN-metadynamics workflow (hen egg white lysozyme)

End-to-end mirror of the [FRESEAN-metadynamics](https://github.com/HeydenLabASU-collab/FRESEAN-metadynamics) `protocol_FRESEAN+singleWTMetad` pipeline for HEWL.

| Step | What happens | This notebook |
|------|--------------|---------------|
| Prep + equilibration | solvate, ions, EM, NPT equilibration | GROMACS instructions (§0) |
| Unbiased MD | 20 ns NPT sampling TRR | load protein trajectory (§1) |
| Coarse-graining | backbone / sidechain COM beads | `CoarseGrain` (§2–3) |
| FRESEAN | correlation matrix + modes | `FRESEAN` on CG trajectory (§4) |
| Back-mapping | CG modes → all-atom | `backmap_modes()` (§5) |
| Mode projection | PLUMED `DIRECTION` PDB + driver | `write_plumed_mode_input()` (§6) |
| Metadynamics | WT-metad in mode space | GROMACS + PLUMED instructions (§7) |
| Reweighting | hills → new CVs | PLUMED instructions (§8) |

You need `input_data/HEWL/1hel.pdb` (included) and the PLUMED/GROMACS templates in `fresean_metaD_data/`. The unbiased MD trajectory from §0 is not in the repo — run the GROMACS steps below or place your own files under `input_data/HEWL/`.

Demo runs use a frame subset; production analyses often use on the order of `10^6` frames.


In [ ]:
import shutil
from pathlib import Path

import matplotlib.pyplot as plt
import MDAnalysis as mda
import numpy as np
import pandas as pd

from pyfresean import Align, CoarseGrain, FRESEAN
from pyfresean.postprocess import low_frequency_peaks, mode_spectra, plot_spectra

METAD_DATA = Path("fresean_metaD_data")
INPUT_DATA = Path("input_data")
INPUT_DATA.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA = Path("output_data")
OUTPUT_DATA.mkdir(parents=True, exist_ok=True)
HEWL = INPUT_DATA / "HEWL"
OUT = OUTPUT_DATA / "HEWL-CG"
OUT.mkdir(parents=True, exist_ok=True)

for name in (
    "plumed-mode-projection.dat",
    "plumed-mode-metadyn.dat",
    "plumed-reweight-CV.dat",
    "plumed-mass+charge.dat",
    "metadyn.mdp",
    "static.job",
):
    shutil.copy2(METAD_DATA / name, OUT / name)

# Metadynamics collective variables: modes 7 and 8 (1-based), zero-frequency bin
MODE_NUMBERS = (7, 8)
MODE_INDICES = tuple(m - 1 for m in MODE_NUMBERS)

print(f"HEWL data directory: {HEWL.resolve()}")
print(f"Output directory: {OUT.resolve()}")
print(f"Using modes {MODE_NUMBERS} (0-based indices {MODE_INDICES})")


## 0. GROMACS setup and unbiased MD

GROMACS + PLUMED (`gmx_plumed`). Starting structure: `input_data/HEWL/1hel.pdb` (PDB ID 1HEL).

**System preparation** — solvate, ions, energy minimization:

```bash
gmx pdb2gmx -f 1hel.pdb -o processed.gro -p topol.top -water tip3p -ff amber99sb-ildn
gmx editconf -f processed.gro -o newbox.gro -c -d 1.0 -bt cubic
gmx solvate -cp newbox.gro -cs spc216.gro -o solv.gro -p topol.top
gmx grompp -f em.mdp -c solv.gro -p topol.top -o ions.tpr
gmx genion -s ions.tpr -o solv_ions.gro -p topol.top -pname NA -nname CL -neutral
gmx mdrun -v -deffnm em
```

**Equilibration** (`equi.mdp`, 100 ps NPT) then **production** (`sample-NPT.mdp`, 20 ns NPT, 20 fs output):

```bash
gmx grompp -f equi.mdp -c em.gro -p topol.top -o equi.tpr
gmx mdrun -v -deffnm equi
gmx grompp -f sample-NPT.mdp -c equi.gro -p topol.top -o sample-NPT.tpr
gmx mdrun -v -deffnm sample-NPT
```

**Protein-only, PBC-corrected trajectory** (for coarse-graining):

```bash
gmx trjconv -s sample-NPT.tpr -f sample-NPT.trr -o sample-NPT_prot_pbc.trr -pbc mol
# select Protein
gmx convert-tpr -s sample-NPT.tpr -o topol_prot.tpr
# select Protein
```

Save `topol_prot.tpr` and `sample-NPT_prot_pbc.trr` under `input_data/HEWL/` before continuing to §1.


## 1. Load all-atom protein trajectory

Files expected under `input_data/HEWL/`:

- `topol_prot.tpr`
- `sample-NPT_prot_pbc.trr`


In [ ]:
topol = HEWL / "topol_prot.tpr"
traj = HEWL / "sample-NPT_prot_pbc.trr"

for path in (topol, traj, HEWL / "1hel.pdb"):
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {path}. Run §0 (GROMACS) or copy the HEWL trajectory into {HEWL}."
        )

u_aa = mda.Universe(str(topol), str(traj))
protein = u_aa.select_atoms("protein")

N_FRAMES = 5000  # increase for production (protocol default: ~1e6)
DT = 0.02  # ps between frames (sample-NPT.mdp: dt=2 fs, output every 10 steps)

print(f"Protein atoms: {protein.n_atoms}")
print(f"Residues: {protein.n_residues}")
print(f"Trajectory: {len(u_aa.trajectory)} frames (using first {N_FRAMES})")


## 2. Coarse-grain mapping

Each canonical residue maps to a **BACK** (backbone COM) bead and, except GLY/ACE/NME, a **SIDE** (sidechain COM) bead.


In [ ]:
cg = CoarseGrain.from_atomgroup(protein)
m = cg.mapping

rows = []
for i in range(m.n_beads):
    rows.append(
        {
            "bead": i,
            "type": m.bead_types[i],
            "resname": m.resnames[i],
            "resindex": int(m.resindices[i]),
            "mass": m.bead_masses[i],
            "n_atoms": len(m.atom_indices[i]),
        }
    )

print(f"CG beads: {m.n_beads}")
print(f"n_constraints: {m.n_constraints}")
pd.DataFrame(rows).head(10)


## 3. Coarse-grain trajectory and reference structures

Outputs go to `output_data/HEWL-CG/`: `topol-cg.top`, `topol-cg.gro`, `traj-cg.trr`, plus `ref.pdb` (all-atom) and `ref-cg.pdb` (first CG frame for alignment).


In [ ]:
cg_top = OUT / "topol-cg.top"
cg_gro = OUT / "topol-cg.gro"
cg_traj = OUT / "traj-cg.trr"
ref_aa_pdb = OUT / "ref.pdb"
ref_cg_pdb = OUT / "ref-cg.pdb"

USE_DISK = True  # stream to disk for long trajectories (recommended for HEWL)

if USE_DISK:
    u_cg = cg.cg_universe(
        stop=N_FRAMES,
        output_cg_topology=str(cg_top),
        output_cg_trajectory=str(cg_traj),
        in_memory=False,
    )
else:
    u_cg = cg.cg_universe(stop=N_FRAMES)
    cg.write_topology(u_cg, str(cg_top))
    cg.write_trajectory(u_cg, str(cg_traj))

u_cg.trajectory[0]
ref_cg = u_cg.atoms.positions.copy()

protein.write(str(ref_aa_pdb))
with open(ref_aa_pdb, "r+", encoding="utf-8") as handle:
    content = handle.read()
    handle.seek(0)
    if not content.startswith("REMARK TYPE=OPTIMAL"):
        handle.write("REMARK TYPE=OPTIMAL\n" + content)

cg.write_topology(u_cg, str(ref_cg_pdb))

print(f"CG beads: {u_cg.atoms.n_atoms}, frames: {len(u_cg.trajectory)}")


## 4. Align CG trajectory and run FRESEAN

Same parameters as the published protocol: `n_corr=100`, `sigma=10` cm⁻¹, alignment to the first CG frame.


In [ ]:
n_corr = 100
sigma = 10.0

u_cg.trajectory.add_transformations(
    Align(u_cg.atoms, reference_positions=ref_cg, place_com_in_box=False),
)

analysis_cg = FRESEAN(
    u_cg,
    select="all",
    n_constraints=cg.mapping.n_constraints,
    n_corr=n_corr,
    dt=DT,
    sigma=sigma,
)
analysis_cg.run(stop=N_FRAMES)

freqs = analysis_cg.results.freqs
vdos = analysis_cg.results.vdos_total
eigenvalues = analysis_cg.results.eigenvalues
eigenvectors = analysis_cg.results.eigenvectors
corr = analysis_cg.results.corr_matrix
avg_temp = analysis_cg.results.avg_temperature

low_peaks = low_frequency_peaks(freqs, vdos, max_freq=200.0)
zero_freq_idx = int(low_peaks[0])

if max(MODE_INDICES) >= eigenvectors.shape[1]:
    raise ValueError(
        f"Modes {MODE_NUMBERS} need at least {max(MODE_NUMBERS)} modes; "
        f"only {eigenvectors.shape[1]} available. Use more frames or check the trajectory."
    )

print(f"T = {avg_temp:.1f} K")
print(f"CG beads = {u_cg.atoms.n_atoms}")
print(f"frequency bin for mode extraction: {zero_freq_idx} ({freqs[zero_freq_idx]:.2f} cm-1)")


### 4a. Spectral plots


In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
plot_spectra(
    freqs,
    vdos,
    ax=ax,
    xlim=(0, 200),
    labels="Total VDoS (CG)",
    vlines=freqs[low_peaks].tolist(),
)
plt.tight_layout()
plt.show()

n_modes = min(8, eigenvectors.shape[1])
mode_vdos = mode_spectra(corr, eigenvectors[zero_freq_idx, :n_modes])
fig, ax = plt.subplots(figsize=(6, 4))
for i in range(n_modes):
    plot_spectra(freqs, mode_vdos[i], ax=ax, labels=f"mode {i + 1}")
ax.set_xlim(0, 200)
plt.tight_layout()
plt.show()


## 5. Back-map modes 7 and 8 to all-atom coordinates

Mass-weighted backmap: `u_i = u_b * sqrt(m_i / M_b)` for each atom `i` in bead `b`.


In [ ]:
aa_modes = []
for mode_num, mode_idx in zip(MODE_NUMBERS, MODE_INDICES):
    cg_mode = eigenvectors[zero_freq_idx, mode_idx]
    aa_mode = cg.backmap_modes(cg_mode)
    aa_modes.append(aa_mode)
    print(f"Mode {mode_num}: CG norm={np.linalg.norm(cg_mode):.4f}, max |AA|={np.max(np.linalg.norm(aa_mode, axis=1)):.4f}")


## 6. PLUMED mode input and projection

`write_plumed_mode_input()` writes `plumed-mode-input.pdb` (reference structure plus modes 7 and 8).

Run the PLUMED driver on the unbiased protein trajectory:

```bash
cd output_data/HEWL-CG
plumed driver --mf_trr input_data/HEWL/sample-NPT_prot_pbc.trr \
  --plumed plumed-mode-projection.dat --kt 2.494339 > plumed-driver.out
```

Estimate Gaussian widths for metadynamics from the projection standard deviations:

```python
proj = np.loadtxt("plumed-mode-projection.out", usecols=[1, 2])
print("std mode 7 (nm):", proj[:, 0].std())
print("std mode 8 (nm):", proj[:, 1].std())
```


In [ ]:
plumed_input = OUT / "plumed-mode-input.pdb"
cg.write_plumed_mode_input(aa_modes, str(plumed_input))

for mode_num, aa_mode in zip(MODE_NUMBERS, aa_modes):
    cg.write_plumed_direction_pdb(aa_mode, str(OUT / f"evec_{mode_num}_aa_scaled.pdb"))

print(f"Wrote {plumed_input}")


## 7. Well-tempered metadynamics — GROMACS

```bash
cd output_data/HEWL-CG
gmx_plumed grompp -f metadyn.mdp -c input_data/HEWL/equi.gro -p input_data/HEWL/topol.top -o metadyn.tpr
gmx_plumed mdrun -v -deffnm metadyn -plumed plumed-mode-metadyn.dat
plumed sum_hills --hills plumed-mode-metadyn.hills \
  --outfile plumed-mode-metadyn.fes --mintozero --kt 2.494339 --stride 5000
```

Set `SIGMA` in `plumed-mode-metadyn.dat` from the projection standard deviations in §6. These steps are not run in the notebook.


## 8. Reweighting — PLUMED

```bash
gmx_plumed mdrun -s metadyn_prot.tpr -nsteps 1 -plumed plumed-mass+charge.dat
gmx_plumed make_ndx -f metadyn_prot.tpr -o groups.ndx
plumed driver --mf_trr metadyn_prot_pbc.trr \
  --plumed plumed-reweight-CV.dat --kt 2.494339 --mc mass+charge.dat > reweight.out
```

Edit `plumed-reweight-CV.dat` for your target collective variables. Not run in the notebook.
